# Policy Gradient Methods: REINFORCE

Instead of learning a value function, **policy gradient** methods directly optimise the policy $\pi_\theta(a|s)$. This notebook:
1. Derives the **policy gradient theorem**
2. Implements **REINFORCE** (Monte Carlo policy gradient)
3. Adds a **baseline** to reduce variance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.distributions import Categorical
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print('PyTorch not installed')

try:
    import gymnasium as gym
    HAS_GYM = True
except ImportError:
    HAS_GYM = False
    print('gymnasium not installed')

%matplotlib inline

## 1. Policy Gradient Theorem

We parameterise the policy $\pi_\theta$ and maximise expected return:
$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_t \gamma^t r_t \right]$$

The gradient is:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t \right]$$

where $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$ is the **return-to-go**.

**REINFORCE** estimates this gradient from a single episode (Monte Carlo).

In [ ]:
if HAS_TORCH:
    class PolicyNetwork(nn.Module):
        def __init__(self, state_dim, action_dim, hidden=128):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(state_dim, hidden),
                nn.ReLU(),
                nn.Linear(hidden, action_dim),
                nn.Softmax(dim=-1)
            )
        
        def forward(self, x):
            return self.net(x)
        
        def act(self, state):
            state_t = torch.FloatTensor(state).unsqueeze(0)
            probs = self.forward(state_t)
            dist = Categorical(probs)
            action = dist.sample()
            return action.item(), dist.log_prob(action)
    
    print('PolicyNetwork defined.')

In [ ]:
if HAS_TORCH and HAS_GYM:
    def reinforce(env_name='CartPole-v1', episodes=500, gamma=0.99, lr=1e-3):
        env = gym.make(env_name)
        state_dim = env.observation_space.shape[0]
        action_dim = env.action_space.n
        
        policy = PolicyNetwork(state_dim, action_dim)
        optimizer = optim.Adam(policy.parameters(), lr=lr)
        rewards_history = []
        
        for ep in range(episodes):
            state, _ = env.reset()
            log_probs = []
            rewards = []
            
            for t in range(500):
                action, log_prob = policy.act(state)
                state, reward, terminated, truncated, _ = env.step(action)
                log_probs.append(log_prob)
                rewards.append(reward)
                if terminated or truncated:
                    break
            
            # Compute returns-to-go
            returns = []
            G = 0
            for r in reversed(rewards):
                G = r + gamma * G
                returns.insert(0, G)
            returns = torch.FloatTensor(returns)
            # Normalise returns (baseline trick)
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
            
            # Policy gradient loss
            loss = 0
            for log_prob, G_t in zip(log_probs, returns):
                loss -= log_prob * G_t
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            rewards_history.append(sum(rewards))
            if (ep + 1) % 100 == 0:
                avg = np.mean(rewards_history[-100:])
                print(f"Episode {ep+1:3d}  avg_reward={avg:.1f}")
        
        env.close()
        return policy, rewards_history
    
    policy, rewards = reinforce()

In [ ]:
if HAS_TORCH and HAS_GYM:
    smoothed = np.convolve(rewards, np.ones(30)/30, mode='valid')
    plt.plot(smoothed)
    plt.axhline(475, color='r', linestyle='--', label='Solved threshold')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward (smoothed)')
    plt.title('REINFORCE on CartPole-v1')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
if HAS_TORCH and HAS_GYM:
    # Compare with and without baseline
    def reinforce_no_baseline(env_name='CartPole-v1', episodes=500, gamma=0.99, lr=1e-3):
        env = gym.make(env_name)
        policy = PolicyNetwork(env.observation_space.shape[0], env.action_space.n)
        optimizer = optim.Adam(policy.parameters(), lr=lr)
        rewards_history = []
        for ep in range(episodes):
            state, _ = env.reset()
            log_probs, rewards_ep = [], []
            for t in range(500):
                action, log_prob = policy.act(state)
                state, reward, term, trunc, _ = env.step(action)
                log_probs.append(log_prob)
                rewards_ep.append(reward)
                if term or trunc: break
            returns = []
            G = 0
            for r in reversed(rewards_ep):
                G = r + gamma * G
                returns.insert(0, G)
            returns = torch.FloatTensor(returns)
            # NO normalisation
            loss = sum(-lp * g for lp, g in zip(log_probs, returns))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            rewards_history.append(sum(rewards_ep))
        env.close()
        return rewards_history
    
    rewards_no_bl = reinforce_no_baseline()
    
    fig, ax = plt.subplots()
    for data, label in [(rewards, 'With baseline'), (rewards_no_bl, 'No baseline')]:
        sm = np.convolve(data, np.ones(30)/30, mode='valid')
        ax.plot(sm, label=label)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward (smoothed)')
    ax.set_title('Effect of Baseline on REINFORCE')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Key Takeaways

- **REINFORCE** directly optimises the policy using the policy gradient theorem.
- It is **on-policy** and uses **Monte Carlo** returns (full episodes).
- A **baseline** (e.g., mean return normalisation) dramatically reduces variance.
- Extensions: Actor-Critic (learned baseline), PPO, A3C.

This completes our tour of core RL algorithms: value iteration, Q-learning, DQN, and policy gradients.